# Esse é o tratamento de todos os arquivos


## Carregando as variáveis de ambiente
*Estou carregando pelo pandas e não pelo numpy. Por que? Porque eu quero!*

In [1]:
import pandas as pd
import numpy as np
import filter as ft
import utils2 as ut
import analysis as an
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from configuration_new import Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## Criando as variáveis iniciais

In [2]:
cfg = Configuration()
path_data = "data/"
path_output =  'output'
path_output =  f"output_{datetime.now():%Y_%m_%d_%H_%M_%S}/"
output = Path(path_output)
# output_directories = {"integration":None, "quality":None, "analysis":None, "sensitivity":None}
# for key in output_directories:
#     output_directories[key]=output/key
#     output_directories[key].mkdir(parents=True, exist_ok=True)

results = {}
filtered_data = {}
quality_control, profiles_rows, sampling_verification = [], [], []

## Carregando arquivos e informações do dataset (metadados)

In [3]:
metadata = ut.get_first_metadata(path_data)
files, failures_files = ut.discover_eeg_txt_files(path_data)

## Fazendo a análise dos arquivos

In [ ]:
for file in files:
    try:
        subject_id, subject_state = file.name.split('.')[0].split('_')

        data_raw = pd.read_csv(file, sep=" ", names=cfg.electrode_labels)

        data_raw = np.transpose(data_raw.to_numpy().astype(float)) #Convertendo os dados para np.ndarray
        
        data_np, changed_values_on_wraparound = ft.fix_integer_wraparound(data_raw)

        data_np = np.ascontiguousarray(data_np)

        data_np, filter_info = ft.apply_filter(data_np, cfg) 

        filtered_data[subject_id] = data_np

        profile = ut.spectral_profile_blocked(data_np, cfg.sampling_frequency, cfg.block_time)

        profile[subject_id] = subject_id

        profile['task'] = subject_state

        profiles_rows.append(profile)

        metrics = ut.quality_control_metrics(data_raw, data_np, filtering_info=filter_info, subject_id=subject_id, file_name=file.name, eeg_task=subject_state, n_wraparound=changed_values_on_wraparound)

        quality_control.append(metrics)
        
    except Exception as error:
        print(f"Failed to process file {file.name}: {error}")
        failures_files.append({
                "file": file.name, 
                "subject_id": file.stem.split("_")[0],
                "stage": "read_or_profile",
                 "error": "%s: %s" % (type(error).__name__, str(error)[:200])})

# Calculando a Coorte nos blocos terminais

In [ ]:
# print(profiles_rows)
cohort_reference = an.cohort_terminal_profile(profiles_rows, cfg)
print(cohort_reference) # Ver essa diferença - possivelmente é o task=still

[0.52443225 0.14601433 0.19590037 0.11376119 0.01989186]


In [25]:
print(profiles_rows[11])
# for p in profiles_rows:
#     print(p.shape)

    block_index  t_start_s   rms_counts  ocular_index  muscle_ratio  prof_delta  prof_theta  prof_alpha  prof_beta  prof_gamma_low  02010013   task
0             0        0.0   968.003171      0.775312      0.017745    0.797228    0.094077    0.074790   0.028688        0.005217  02010013  still
1             1       20.0   547.150968      0.426792      0.038794    0.499162    0.161076    0.262073   0.065257        0.012433  02010013  still
2             2       40.0   464.263518      0.330102      0.048544    0.388543    0.210379    0.307877   0.076644        0.016557  02010013  still
3             3       60.0   520.272640      0.503627      0.042135    0.579249    0.128581    0.214694   0.064434        0.013043  02010013  still
4             4       80.0   861.209124      0.698645      0.016271    0.720887    0.126545    0.117729   0.029482        0.005357  02010013  still
5             5      100.0   532.730657      0.381220      0.039858    0.459263    0.172969    0.279882   0.0756